In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
from start_line.plotting import *
from concept_abstraction.environments import Cyclic4StateEnv, TreeRepeatEnv
from concept_abstraction.training import train_model
from concept_abstraction.selection import greedy_selection, random_selection, human_centered_selection
from concept_abstraction.env_utils import *
import torch
import sys 
import argparse
import secrets
import numpy as np 
import random 

In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [4]:
if is_jupyter: 
    seed        = 43
    environment_string = "tree"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string}

In [6]:
np.random.seed(seed)
random.seed(seed)

## Concept Baseline

In [7]:
baseline_concepts = get_baseline_concept_sets(environment_string)

In [9]:
values_by_concept = []
for concept_list in baseline_concepts:
    env = create_environment_from_string(environment_string,concept_list,0)
    q_net = train_model(env)
    values_by_concept.append(get_values(env,q_net))

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/training.py:34: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  torch.tensor(obs, dtype=torch.float32),


## Concept Selection

In [11]:
selected_concepts = []
values_by_random_concept = []
for k in range(1,round(len(env.concepts)**0.5)+1):
    random_concepts = random_selection(env,k)
    env = create_environment_from_string(environment_string,random_concepts,0)
    q_net = train_model(env)
    selected_concepts.append(random_concepts)
    values_by_random_concept.append(get_values(env,q_net))

In [15]:
selected_concepts = []
values_by_random_concept = []
for k in range(1,round(len(env.concepts)**0.5)):
    greedy_concepts = greedy_selection(env,k)
    env = create_environment_from_string(environment_string,greedy_concepts,0)
    q_net = train_model(env)
    selected_concepts.append(greedy_concepts)
    values_by_random_concept.append(get_values(env,q_net))

In [22]:
baseline_concepts

[[0, 1, 2, 3], [4]]

In [51]:
concept_list = list(range(env.concepts.shape[0]+1))
accuracy_by_concept = np.random.random(len(concept_list))
target_abstraction = np.random.random()*0.25
max_concept = 3

selected_concepts = human_centered_selection(env,accuracy_by_concept,target_abstraction)
selected_concepts = [concept_list[idx] for idx,i in enumerate(selected_concepts) if i>=0.5]
selected_accuracies = [accuracy_by_concept[idx] for idx,i in enumerate(selected_concepts) if i>=0.5]

env = create_environment_from_string(environment_string,selected_concepts,1-np.mean(selected_accuracies))
q_net = train_model(env)
human_perf = get_values(env,q_net)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-7700K CPU @ 4.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 51 rows, 7 columns and 206 nonzeros
Model fingerprint: 0x24419e88
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e-02, 8e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 29 rows and 0 columns
Presolve time: 0.00s
Presolved: 22 rows, 7 columns, 80 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.2498723e-01   1.000000e+00   0.000000e+00      0s
       5    6.3284516e-01   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.328451630e-01


## Performance under Uncertainty

In [16]:
epsilons = [0,0.01,0.1,0.25,0.5]
values_error = [[[] for _ in baseline_concepts[1:]] for _ in epsilons]

for idx,e in enumerate(epsilons):
    for jdx,concept_list in enumerate(baseline_concepts[1:]):
        env = create_environment_from_string(environment_string,concept_list,e)
        q_net = train_model(env)
        values_error[idx][jdx] = get_values(env,q_net)

In [ ]:
# TODO: 1) Double check on the tree environment, 2) Save all the data, 3) Add comments + clean up functions